# Real vs simulated vs generated events

One recorded human action as three kinds of event data: real DAVIS346 events, events simulated from a 240 fps phone video (ESIM-style, notebook 06) and event images generated by diffusion from the person's 2D pose (ControlEvents, notebook 09). No training, no new methods.

Generation needs a GPU: use a Colab GPU runtime (on Apple silicon each image takes about 30 s). Generated images and keypoints are cached next to the data, so later runs only redo the comparison. Sections whose data is missing print `SKIPPED: <reason>`.

## Setup

In [1]:
import os, sys
if "google.colab" in sys.modules:
    REPO = "/content/event-perception"
    if not os.path.exists(REPO):
        !git clone https://github.com/allusson/event-perception.git {REPO}
    %cd {REPO}
    !pip install -q -e ".[diffusion]"
    sys.path.insert(0, REPO)
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/event-perception-data/human"
else:
    # local: assumes `pip install -e ".[diffusion]"` was run from the repo root
    os.chdir(os.path.dirname(os.path.abspath("")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
    DATA_DIR = "data/human"

In [2]:
import dataclasses, time, warnings
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display
from PIL import Image
from diffusers.utils import logging as diffusers_logging
diffusers_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

from evcam import controlevents as ce
from evcam.sources import FFMPEG_HINT, load_davis, load_video
from evcam.simulate import simulate_events
from evcam.representations import event_image
from evcam.viz import show_events

DEVICE, DTYPE = ce.pick_device()
SIZE = ce.SIZE                       # generation canvas, px
print(f"device {DEVICE}, data in {DATA_DIR}")


def disp(img):
    # display only: linear intensity -> gamma 1/2.2
    return np.clip(img, 0, 1) ** (1 / 2.2)


def in_window(ev, t0_us, t1_us):
    return ev[(ev["t"] >= t0_us) & (ev["t"] < t1_us)]


def count_image(ev, wh):
    """Events per pixel, polarity ignored."""
    img = np.zeros((wh[1], wh[0]), np.float32)
    np.add.at(img, (ev["y"], ev["x"]), 1)
    return img


def white_on_black(img):
    """Scale to [0, 1] by the 99th percentile of the non-zero pixels: white events on black."""
    v = np.percentile(img[img > 0], 99) if (img > 0).any() else 1.0
    return np.clip(img / v, 0, 1)


T_NOTEBOOK = time.time()

device mps, data in data/human


## 0. Parameters

Motion-start values and preview starts are per action and per camera, read off the previews in section 2, as in notebook 06. Crop boxes are `(x0, y0, x1, y1)` around the person: `VIDEO_CROP` in phone pixels (full resolution), `DAVIS_CROP` in DAVIS pixels; `None` uses the centred square of the frame. The canvas is the square around the crop, resized to 512 × 512.

In [3]:
ACTIONS = ("wave", "legraise", "walk")
ACTION = "wave"
CAPTURE_FPS = 240          # true capture rate of the phone clip
C_SIM = 0.2                # contrast threshold of the simulation
GEN_FPS = 10               # rate of the compared frames, Hz
ACC_MS = 15                # real and simulated events accumulated over this window, centred on each frame time
CLIP_S = 3.0               # analysed span after the motion start, s

VIDEO_START_FRAME = {"wave": None, "legraise": None, "walk": None}       # first phone frame where the person moves
DAVIS_START_MS = {"wave": None, "legraise": None, "walk": None}          # start of the first DAVIS window that lights up, ms
VIDEO_PREVIEW_START_S = {"wave": 0.5, "legraise": 0.5, "walk": 0.5}     # phone preview start, s on the phone clock
DAVIS_PREVIEW_START_S = {"wave": 0.5, "legraise": 0.5, "walk": 0.5}     # DAVIS preview start, s on the DAVIS clock
VIDEO_CROP = {"wave": None, "legraise": None, "walk": None}
DAVIS_CROP = {"wave": None, "legraise": None, "walk": None}

NUM_STEPS = 30             # UniPC steps
GUIDANCE = 7.5             # classifier-free guidance scale
SEED = 0
REGENERATE = False         # True: recompute keypoints and images even if cached

VIDEO_LONG_SIDE = 480      # phone frames are loaded at this long side (px) for the previews and the simulation
SIM_MARGIN_S = 0.2         # simulated before and after the analysed span

## 1. Idea

Three ways to get event data of the same motion:

- **Real**: record it with the DAVIS346.
- **Simulated**: from high frame rate video with the event generation model of notebook 06. Physics: it models every pixel's log intensity, so it needs a high frame rate.
- **Generated**: from the 2D pose with the ControlEvents skeleton model (notebook 09). It needs only a skeleton and has learned from data what events of a moving person look like, but knows nothing about the actual brightness or timing.

All three are compared as event images. Real and simulated events are counted per pixel over `ACC_MS` around each frame time, without polarity (white events on black), which is the format of the skeleton model's output: it was trained on DHP-19 and CDEHP event images rendered without polarity.

## 2. Load and align

The phone clip is read with `load_video(..., capture_fps=CAPTURE_FPS)` (iPhone slow-motion files store a 30 fps playback rate) at `VIDEO_LONG_SIDE`, linear intensity, aspect kept. The two cameras start recording at different times, so the clips are aligned on the motion start, picked by hand as in notebook 06. The first figure covers each whole clip: DAVIS event rate (10 ms bins) and the phone's mean absolute difference between consecutive frames; the motion starts at the first big jump. Set `VIDEO_PREVIEW_START_S` and `DAVIS_PREVIEW_START_S` just before it. The previews then show 40 phone frames and 40 consecutive 10 ms DAVIS event images from there: enter the first phone frame where the person has moved as `VIDEO_START_FRAME` and the first DAVIS window that lights up as `DAVIS_START_MS`, and re-run. The last figure has pixel axes for the crop boxes. The compared frame times are then `GEN_FPS` per second over `CLIP_S` after the motion start.

In [4]:
t_sec = time.time()
VIDEO_PATH = next((f"{DATA_DIR}/{ACTION}{e}" for e in (".mov", ".MOV", ".mp4", ".MP4")
                   if os.path.exists(f"{DATA_DIR}/{ACTION}{e}")), None)
DAVIS_PATH = f"{DATA_DIR}/{ACTION}.aedat4"
CACHE = f"{DATA_DIR}/cache/{ACTION}"
video = davis = None
skip = None                                   # reason the later sections are skipped
if VIDEO_PATH is None:
    skip = f"no {DATA_DIR}/{ACTION}.mov or .mp4"
elif not os.path.exists(DAVIS_PATH):
    skip = f"{DAVIS_PATH} not found"
if skip:
    print(f"SKIPPED: {skip}")
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    ok, first = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"OpenCV cannot decode {VIDEO_PATH}. Convert it first: {FFMPEG_HINT}")
    PHONE_WH = (first.shape[1], first.shape[0])                        # full resolution, after rotation
    s = VIDEO_LONG_SIDE / max(PHONE_WH)
    video = load_video(VIDEO_PATH, capture_fps=CAPTURE_FPS, resize_to=(round(PHONE_WH[1] * s), round(PHONE_WH[0] * s)))
    davis = load_davis(DAVIS_PATH)
    H, W = video.shape
    Hd, Wd = davis.shape
    print(f"phone: {PHONE_WH[0]}x{PHONE_WH[1]}, {video.n_frames} frames ({video.n_frames / CAPTURE_FPS:.2f} s), loaded at {W}x{H}")
    print(f"DAVIS: {davis.summary()}")

    fig, ax = plt.subplots(1, 2, figsize=(14, 2.8))
    edges = np.arange(0, davis.events["t"][-1] + 10000, 10000)
    ax[0].plot(edges[:-1] / 1e6, np.histogram(davis.events["t"], edges)[0] / 1e4, lw=0.8)   # per 10 ms -> M / s
    ax[0].set_xlabel("DAVIS time (s)"); ax[0].set_ylabel("M events / s"); ax[0].set_title("DAVIS event rate, 10 ms bins")
    mad = [np.abs(video.frames[k + 1] - video.frames[k]).mean() for k in range(video.n_frames - 1)]
    ax[1].plot(video.frame_t[1:] / 1e6, mad, lw=0.8)
    ax[1].set_xlabel("phone time (s)"); ax[1].set_ylabel("mean |frame difference|"); ax[1].set_title("phone, consecutive frames")
    plt.tight_layout(); plt.show()

    N_PREVIEW, DAVIS_STEP_MS = 40, 10
    k0 = int(round(VIDEO_PREVIEW_START_S[ACTION] * CAPTURE_FPS))
    ks = range(k0, min(k0 + N_PREVIEW, video.n_frames))
    fig, ax = plt.subplots(5, 8, figsize=(16, 8.5))
    for a, k in zip(ax.ravel(), ks):
        a.imshow(disp(video.frames[k]), cmap="gray", vmin=0, vmax=1)
        a.set_title(f"frame {k}, {video.frame_t[k] / 1e6:.3f} s", fontsize=7)
    for a in ax.ravel(): a.axis("off")
    fig.suptitle(f"phone frames from {VIDEO_PREVIEW_START_S[ACTION]} s"); plt.tight_layout(); plt.show()

    t_p0 = int(round(DAVIS_PREVIEW_START_S[ACTION] * 1e6))
    starts = t_p0 + np.arange(N_PREVIEW) * DAVIS_STEP_MS * 1000
    fig, ax = plt.subplots(5, 8, figsize=(16, 8.5))
    for a, t in zip(ax.ravel(), starts):
        show_events(a, event_image(in_window(davis.events, t, t + DAVIS_STEP_MS * 1000), (Wd, Hd, 2)),
                    colorbar=False, title=f"{t / 1e3:.0f} ms", fontsize=7)
        a.axis("off")
    fig.suptitle(f"DAVIS events, consecutive {DAVIS_STEP_MS} ms windows from {DAVIS_PREVIEW_START_S[ACTION]} s, "
                 "ON − OFF (red = ON, blue = OFF)")
    plt.tight_layout(); plt.show()

    span = in_window(davis.events, starts[0], starts[-1] + DAVIS_STEP_MS * 1000)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].imshow(disp(video.frames[ks[-1]]), cmap="gray", vmin=0, vmax=1, extent=(0, PHONE_WH[0], PHONE_WH[1], 0))
    ax[0].set_title(f"phone frame {ks[-1]}, full-resolution pixels (for VIDEO_CROP)")
    show_events(ax[1], event_image(span, (Wd, Hd, 2)), title="DAVIS events over the preview span (for DAVIS_CROP)")
    plt.tight_layout(); plt.show()

    if VIDEO_START_FRAME[ACTION] is None or DAVIS_START_MS[ACTION] is None:
        skip = "motion start not set"
        print(f'Set VIDEO_START_FRAME["{ACTION}"] and DAVIS_START_MS["{ACTION}"] in section 0 from the previews above, '
              "then re-run.")

if not skip:
    t_video0 = int(video.frame_t[VIDEO_START_FRAME[ACTION]])
    t_davis0 = int(round(DAVIS_START_MS[ACTION] * 1e3))
    real_ev = davis.events.copy(); real_ev["t"] -= t_davis0                         # time since motion start, us
    frame_us = np.round(np.arange(int(round(CLIP_S * GEN_FPS))) * 1e6 / GEN_FPS).astype(np.int64)
    frame_idx = VIDEO_START_FRAME[ACTION] + np.round(frame_us * CAPTURE_FPS / 1e6).astype(int)
    n = int(np.sum((frame_idx < video.n_frames) & (frame_us + ACC_MS * 500 < real_ev["t"][-1])))
    frame_us, frame_idx = frame_us[:n], frame_idx[:n]
    box_phone = ce.canvas_box(PHONE_WH, VIDEO_CROP[ACTION])                       # full-resolution phone px
    box_work = tuple(int(round(v * W / PHONE_WH[0])) for v in box_phone)          # same box at VIDEO_LONG_SIDE
    box_davis = ce.canvas_box((Wd, Hd), DAVIS_CROP[ACTION])
    print(f"phone: motion starts at frame {VIDEO_START_FRAME[ACTION]} ({t_video0 / 1e3:.1f} ms); "
          f"DAVIS: at {DAVIS_START_MS[ACTION]} ms")
    print(f"{n} frame times, {1000 / GEN_FPS:.0f} ms apart, over {frame_us[-1] / 1e6:.2f} s")
print(f"section 2: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 2: 0.0 s


## 3. Real events

DAVIS events counted per pixel over `ACC_MS` centred on each frame time, polarity ignored, mapped to the canvas (`DAVIS_CROP`). Six of the frame times, evenly spaced, are shown here and in the later figures.

In [5]:
t_sec = time.time()
if skip:
    print(f"SKIPPED: {skip}")
else:
    half = ACC_MS * 500
    real_img = [white_on_black(ce.to_canvas(count_image(in_window(real_ev, t - half, t + half), (Wd, Hd)), box_davis))
                for t in frame_us]
    ROWS = np.unique(np.linspace(0, n - 1, min(6, n)).round().astype(int))
    fig, ax = plt.subplots(1, len(ROWS), figsize=(2.8 * len(ROWS), 3), squeeze=False)
    for a, r in zip(ax[0], ROWS):
        a.imshow(real_img[r], cmap="gray", vmin=0, vmax=1); a.set_title(f"{frame_us[r] / 1e3:.0f} ms", fontsize=10); a.axis("off")
    fig.suptitle(f"real events, {ACC_MS} ms, time since motion start"); plt.tight_layout(); plt.show()
print(f"section 3: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 3: 0.0 s


## 4. Simulated events (ESIM)

`simulate_events` (notebook 06) at $c$ = `C_SIM` on the phone frames of the analysed span plus `SIM_MARGIN_S` on each side, which keeps memory low; the margin before the motion start lets the reference levels settle on the still person. Accumulated as in section 3, in phone coordinates mapped to the canvas (`VIDEO_CROP`).

In [6]:
t_sec = time.time()
if skip:
    print(f"SKIPPED: {skip}")
else:
    i0 = max(frame_idx[0] - int(SIM_MARGIN_S * CAPTURE_FPS), 0)
    i1 = min(frame_idx[-1] + int((SIM_MARGIN_S + ACC_MS / 2000) * CAPTURE_FPS) + 1, video.n_frames)
    sim = simulate_events(dataclasses.replace(video, frames=video.frames[i0:i1], frame_t=video.frame_t[i0:i1]), c=C_SIM)
    sim_ev = sim.events.copy(); sim_ev["t"] -= t_video0
    sim_img = [white_on_black(ce.to_canvas(count_image(in_window(sim_ev, t - half, t + half), (W, H)), box_work))
               for t in frame_us]
    print(f"{sim.n_events:,} events from frames {i0} to {i1 - 1}")
    fig, ax = plt.subplots(1, len(ROWS), figsize=(2.8 * len(ROWS), 3), squeeze=False)
    for a, r in zip(ax[0], ROWS):
        a.imshow(sim_img[r], cmap="gray", vmin=0, vmax=1); a.set_title(f"{frame_us[r] / 1e3:.0f} ms", fontsize=10); a.axis("off")
    fig.suptitle(f"simulated events, c = {C_SIM}, {ACC_MS} ms"); plt.tight_layout(); plt.show()
print(f"section 4: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 4: 0.0 s


## 5. Generated events (ControlEvents)

For each frame time: the phone frame (canvas, colour) → OpenPose → 13 joints → red skeleton on white → ControlEvents skeleton model (`combined_unet_all` + `combined_controlnet_all`), prompt "A moving human", as in notebook 09. Each frame is generated independently, so every frame starts from the same initial noise (a generator seeded with `SEED`): otherwise the appearance changes from frame to frame. Frames where OpenPose finds no person are skipped. Keypoints and images are cached in `<DATA_DIR>/cache/<action>/` and reused unless `REGENERATE` is set; delete the folder after changing the crop or the motion start.

In [7]:
t_sec = time.time()


def read_canvas_frames(path, indices, box):
    """Colour phone frames (RGB uint8, canvas) at the given frame indices, read in order."""
    want, out, k = set(int(i) for i in indices), {}, 0
    cap = cv2.VideoCapture(path)
    ok, bgr = cap.read()
    while ok and k <= max(want):
        if k in want:
            out[k] = ce.to_canvas(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), box)
        ok, bgr = cap.read()
        k += 1
    cap.release()
    return out


def gen_path(i, seed):
    return f"{CACHE}/gen_f{i:05d}_s{seed}.png"


detector = None


def joints_cached(key, rgb):
    """OpenPose 13 joints on the canvas, from the cache if there; all NaN = no person."""
    global detector
    if key not in poses:
        if detector is None:
            detector = ce.openpose_detector()
        xy = ce.detect_joints(detector, rgb)
        poses[key] = np.full((13, 2), np.nan) if xy is None else xy
    return None if np.isnan(poses[key]).all() else poses[key]


if skip:
    print(f"SKIPPED: {skip}")
else:
    os.makedirs(CACHE, exist_ok=True)
    POSE_FILE = f"{CACHE}/pose.npz"
    poses = {} if REGENERATE or not os.path.exists(POSE_FILE) else dict(np.load(POSE_FILE))
    phone_rgb = read_canvas_frames(VIDEO_PATH, frame_idx, box_phone)
    joints = [joints_cached(f"phone_{i}", phone_rgb[i]) for i in frame_idx]
    np.savez(POSE_FILE, **poses)
    has_person = [k for k in range(n) if joints[k] is not None]
    flicker = has_person[:3]                                         # 3 consecutive frames for the figure below
    jobs = [(k, SEED) for k in has_person] + [(k, SEED + j) for j, k in enumerate(flicker) if j > 0]
    todo = [(k, s) for k, s in jobs if REGENERATE or not os.path.exists(gen_path(frame_idx[k], s))]
    if todo:
        t_gen = time.time()
        pipe = ce.control_pipe(ce.SKELETON_UNET, ce.SKELETON_CONTROLNET, DEVICE, DTYPE)
        print(f"model loaded in {time.time() - t_gen:.0f} s")
        t_gen = time.time()
        for k, s in todo:
            ce.generate(pipe, ce.PROMPT_HUMAN, ce.draw_skeleton(joints[k]), seed=s, num_steps=NUM_STEPS,
                        guidance=GUIDANCE).save(gen_path(frame_idx[k], s))
        print(f"{len(todo)} images generated, {(time.time() - t_gen) / len(todo):.1f} s each ({DEVICE})")
        pipe = None; ce.free(DEVICE)
    gen_img = [None if joints[k] is None else
               np.asarray(Image.open(gen_path(frame_idx[k], SEED)).convert("L"), np.float32) / 255 for k in range(n)]
    print(f"{n} frames: {len(has_person)} with a person, {n - len(has_person)} skipped (no person), "
          f"{len(jobs) - len(todo)} of {len(jobs)} images from the cache")
print(f"section 5: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 5: 0.0 s


**Fixed vs changing noise.** Three consecutive frames generated from the same initial noise (top) and with seed `SEED + k` for the k-th frame (bottom).

In [8]:
if skip:
    print(f"SKIPPED: {skip}")
elif len(flicker) < 3:
    print("SKIPPED: fewer than 3 frames with a person")
else:
    fig, ax = plt.subplots(2, 3, figsize=(10, 6.8))
    for j, k in enumerate(flicker):
        for r, s in enumerate((SEED, SEED + j)):
            ax[r, j].imshow(Image.open(gen_path(frame_idx[k], s)))
            ax[r, j].set_title(f"{frame_us[k] / 1e3:.0f} ms, seed {s}", fontsize=10); ax[r, j].axis("off")
    ax[0, 0].text(-0.08, 0.5, "same noise", transform=ax[0, 0].transAxes, rotation=90, va="center", ha="right")
    ax[1, 0].text(-0.08, 0.5, "seed per frame", transform=ax[1, 0].transAxes, rotation=90, va="center", ha="right")
    plt.tight_layout(); plt.show()

SKIPPED: no data/human/wave.mov or .mp4


## 6. Side by side

Rows: six frame times. Columns: phone frame with the OpenPose skeleton, real events, simulated events, generated events. The cameras sit next to each other but are not calibrated, so the views differ slightly. The animation plays all frame times at `GEN_FPS` and is saved as `comparison.gif` in the cache folder.

In [9]:
t_sec = time.time()
COLUMNS = ("phone + skeleton", "real (DAVIS)", f"simulated, c = {C_SIM}", "generated (ControlEvents)")


def tiles(k, size=SIZE):
    """The four images of frame k as (size, size, 3) uint8."""
    phone = phone_rgb[frame_idx[k]]
    phone = np.asarray(ce.draw_skeleton(joints[k], image=phone)) if joints[k] is not None else phone
    gray = lambda g: np.repeat((np.clip(g, 0, 1) * 255).astype(np.uint8)[..., None], 3, axis=2)
    gen = gray(gen_img[k]) if gen_img[k] is not None else np.zeros((SIZE, SIZE, 3), np.uint8)
    return [cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
            for im in (phone, gray(real_img[k]), gray(sim_img[k]), gen)]


if skip:
    print(f"SKIPPED: {skip}")
else:
    fig, ax = plt.subplots(len(ROWS), 4, figsize=(12, 3.1 * len(ROWS)), squeeze=False)
    for r, k in enumerate(ROWS):
        for c, im in enumerate(tiles(k)):
            ax[r, c].imshow(im); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
            if r == 0:
                ax[r, c].set_title(COLUMNS[c], fontsize=10)
        ax[r, 0].set_ylabel(f"{frame_us[k] / 1e3:.0f} ms")
        if gen_img[k] is None:
            ax[r, 3].text(0.5, 0.5, "no person", color="w", ha="center", transform=ax[r, 3].transAxes)
    fig.suptitle(f"{ACTION}: time since motion start, events over {ACC_MS} ms"); plt.tight_layout(); plt.show()

    T = 256
    header = np.full((28, 4 * T, 3), 255, np.uint8)
    for c, name in enumerate(COLUMNS):
        cv2.putText(header, name, (c * T + 6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    frames = []
    for k in range(n):
        row = np.concatenate(tiles(k, T), axis=1)
        cv2.putText(row, f"{frame_us[k] / 1e3:.0f} ms", (6, T - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
        frames.append(Image.fromarray(np.concatenate([header, row], axis=0)))
    GIF = f"{CACHE}/comparison.gif"
    frames[0].save(GIF, save_all=True, append_images=frames[1:], duration=int(1000 / GEN_FPS), loop=0)
    print(f"saved {GIF}")
    display(ShowImage(filename=GIF))
print(f"section 6: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 6: 0.0 s


## 7. Where the events are: arms vs legs

For each method and frame time, the share of event pixels (value above `EVENT_THRESH` in the white-on-black image) within `LIMB_DIST` of the arm segments (shoulder-elbow-hand) and of the leg segments (hip-knee-foot), with the keypoints in each image's own coordinates: phone OpenPose keypoints for the simulated and generated images, OpenPose on the nearest DAVIS APS frame for the real events. For a wave the real events should concentrate on the arm, for a leg raise on the leg; this tests whether the generated events follow which limbs actually move or just the pose.

In [10]:
t_sec = time.time()
EVENT_THRESH = 0.2         # event pixel: value above this in the [0, 1] image
LIMB_DIST = 0.04           # distance to a limb segment, fraction of the canvas side (about 20 px)


def limb_mask(xy, limbs):
    m = np.zeros((SIZE, SIZE), np.uint8)
    p = lambda j: tuple(int(v) for v in xy[j] * SIZE)
    for a, b in limbs:
        if not np.isnan(xy[[a, b]]).any():
            cv2.line(m, p(a), p(b), 1, 2 * int(LIMB_DIST * SIZE))
    return m.astype(bool)


def limb_shares(img, xy):
    """(arm share, leg share) of the event pixels of img; NaN without keypoints or events."""
    if img is None or xy is None:
        return np.nan, np.nan
    ev = img > EVENT_THRESH
    if not ev.any():
        return np.nan, np.nan
    return (ev & limb_mask(xy, ce.ARM_LIMBS)).sum() / ev.sum(), (ev & limb_mask(xy, ce.LEG_LIMBS)).sum() / ev.sum()


if skip:
    print(f"SKIPPED: {skip}")
else:
    real_joints = [None] * n
    if davis.frames is None:
        print(f"SKIPPED (real column): no APS frames in the recording ({davis.meta.get('notes')})")
    else:
        for k, t in enumerate(frame_us):
            j = int(np.argmin(np.abs(davis.frame_t - (t + t_davis0))))
            aps = (np.clip(ce.to_canvas(davis.frames[j], box_davis), 0, 1) * 255).astype(np.uint8)
            real_joints[k] = joints_cached(f"aps_{j}", np.repeat(aps[..., None], 3, axis=2))
        np.savez(POSE_FILE, **poses)
        found = sum(x is not None for x in real_joints)
        print(f"OpenPose on the APS frames: a person in {found} of {n}")
        if not found:
            print("SKIPPED (real column): OpenPose finds no person on the APS frames")

    methods = {"real": (real_img, real_joints), "simulated": (sim_img, joints), "generated": (gen_img, joints)}
    shares = {m: np.array([limb_shares(imgs[k], kps[k]) for k in range(n)]) for m, (imgs, kps) in methods.items()}
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
    for m, sh in shares.items():
        for a, col in zip(ax, (0, 1)):
            a.plot(frame_us / 1e3, sh[:, col], "o-", ms=3, lw=1, label=m)
    for a, name in zip(ax, ("arms", "legs")):
        a.set_xlabel("time since motion start (ms)"); a.set_title(f"share of event pixels near the {name}")
    ax[0].set_ylabel("share"); ax[0].legend(); plt.tight_layout(); plt.show()
    print(f"{'mean share':<12}{'arms':>8}{'legs':>8}")
    for m, sh in shares.items():
        print(f"{m:<12}{np.nanmean(sh[:, 0]) if np.isfinite(sh[:, 0]).any() else np.nan:>8.2f}"
              f"{np.nanmean(sh[:, 1]) if np.isfinite(sh[:, 1]).any() else np.nan:>8.2f}")
print(f"section 7: {time.time() - t_sec:.1f} s")

SKIPPED: no data/human/wave.mov or .mp4
section 7: 0.0 s


In [11]:
print(f"notebook runtime {time.time() - T_NOTEBOOK:.0f} s")

notebook runtime 0 s


## References

1. H. Rebecq, D. Gehrig, D. Scaramuzza, "ESIM: an Open Event Camera Simulator", Conference on Robot Learning (CoRL) 2018.
2. Y. Hu, Y. Xue, S. Klenk, D. Cremers, G. Pons-Moll, "ControlEvents: Controllable Synthesis of Event Camera Data with Foundational Prior from Image Diffusion Models", WACV 2026, [arXiv:2509.22864](https://arxiv.org/abs/2509.22864). Weights: [huggingface.co/EventDiffusion](https://huggingface.co/EventDiffusion) (MIT).
3. R. Rombach, A. Blattmann, D. Lorenz, P. Esser, B. Ommer, "High-Resolution Image Synthesis with Latent Diffusion Models", CVPR 2022 (Stable Diffusion; weights under CreativeML OpenRAIL-M).
4. L. Zhang, A. Rao, M. Agrawala, "Adding Conditional Control to Text-to-Image Diffusion Models", ICCV 2023 (ControlNet).
5. W. Zhao, L. Bai, Y. Rao, J. Zhou, J. Lu, "UniPC: A Unified Predictor-Corrector Framework for Fast Sampling of Diffusion Models", NeurIPS 2023.
6. Z. Cao, G. Hidalgo, T. Simon, S.-E. Wei, Y. Sheikh, "OpenPose: Realtime Multi-Person 2D Pose Estimation Using Part Affinity Fields", IEEE TPAMI 43(1), 2021 (online 2019).
7. E. Calabrese, G. Taverni, C. Awai Easthope, S. Skriabine, F. Corradi, L. Longinotti, K. Eng, T. Delbruck, "DHP19: Dynamic Vision Sensor 3D Human Pose Dataset", CVPR Workshops 2019.
8. Z. Shao, X. Wang, W. Zhou, W. Wang, J. Yang, Y. Li, "A Temporal Densely Connected Recurrent Network for Event-based Human Pose Estimation", Pattern Recognition 147, 2024 (CDEHP).
9. S. Zou, C. Guo, X. Zuo, S. Wang, P. Wang, X. Hu, S. Chen, M. Gong, L. Cheng, "EventHPE: Event-based 3D Human Pose and Shape Estimation", ICCV 2021 (Event-HPE, 15 ms event windows).
10. C. Brandli, R. Berner, M. Yang, S.-C. Liu, T. Delbruck, "A 240×180 130 dB 3 µs Latency Global Shutter Spatiotemporal Vision Sensor", IEEE Journal of Solid-State Circuits 49(10), 2014 (DAVIS sensor).